In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Load environment variables from .env file
env_path = Path("..") / ".env"
load_dotenv(dotenv_path=env_path)

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
if not GOOGLE_API_KEY:
    raise ValueError("GOOGLE_API_KEY not found in .env file")

# Install google-generativeai if not already installed
# !pip install -q -U google-generativeai python-dotenv datasets

In [2]:
import json
from datasets import load_dataset
import google.generativeai as genai
from datetime import datetime
from pathlib import Path

dataset_test = load_dataset("mkieffer/MEDEC", split="test")
genai.configure(api_key=GOOGLE_API_KEY)

/Users/henry/anaconda3/envs/cs_263/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/2p/d0nnr3sx5pq52lv81g8nwn500000gn/T/ipykernel_99168/3204701561.py:3: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [3]:
dataset_test[0]

{'text_id': 'ms-test-0',
 'text': "A 29-year-old internal medicine resident presents to the emergency department with complaints of fevers, diarrhea, abdominal pain, and skin rash for 2 days. He feels fatigued and has lost his appetite. On further questioning, he says that he returned from his missionary trip to Brazil last week. He is excited as he talks about his trip. Besides a worthy clinical experience, he also enjoyed local outdoor activities, like swimming and rafting. His past medical history is insignificant. The blood pressure is 120/70 mm Hg, the pulse is 100/min, and the temperature is 38.3 C (100.9 F). On examination, there is a rash on the legs. Patient's symptoms are suspected to be due to hepatitis A. The rest of the examination is normal.",
 'sentences': "0 | A 29-year-old internal medicine resident presents to the emergency department with complaints of fevers, diarrhea, abdominal pain, and skin rash for 2 days.\n1 | He feels fatigued and has lost his appetite.\n2 | O

In [4]:
def load_indices(path: str):
    return json.loads(Path(path).read_text(encoding="utf-8"))["indices"]

loaded_indices = load_indices("sampled_test_indices.json")
subset_test = dataset_test.select(loaded_indices)

In [5]:
system_instruction = """
The following is a medical narrative about a patient. You are a skilled medical doctor reviewing the clinical text. The text is either correct or contains one error.
The text has a sentence per line. Each line starts with the sentence ID, followed by a pipe character then the sentence to check. Check every sentence of the text.
If the text is correct return the following output: CORRECT. If the text has a medical error, return the sentence id of the sentence containing the error,
followed by a space, and a corrected version of the sentence.
"""

def escape_for_double_quotes(s: str) -> str:
    # Escape backslashes and double quotes so the output stays valid when wrapped in "...".
    return s.replace("\\", "\\\\").replace('"', '\\"')

def parse_model_output(output_text: str):
    """
    Parse the model output into the submission fields.
    """
    t = (output_text or "").strip()
    if t.upper().rstrip(".") == "CORRECT":
        return 0, -1, None
    parts = t.split(None, 1)
    if not parts:
        return 0, -1, None
    try:
        sid = int(parts[0])
    except ValueError:
        return 0, -1, None
    corrected = parts[1].strip() if len(parts) > 1 else ""
    if not corrected:
        corrected = "NA"
    return 1, sid, corrected

# ===== main =====
MODEL_NAME = "gemini-2.5-flash"  # or "gemini-1.5-pro"
model = genai.GenerativeModel(
    model_name=MODEL_NAME,
    system_instruction=system_instruction
)

out_dir = Path("outputs")
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / f"medec-ms_{MODEL_NAME}_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"

num_examples = len(subset_test)
with out_path.open("w", encoding="utf-8") as f:
    for i in range(num_examples):
        item = subset_test[i]
        text_id = item["text_id"]
        user_prompt = item["sentences"]

        try:
            response = model.generate_content(user_prompt)
            output_text = response.text
        except Exception as e:
            print(f"Error generating content for {text_id}: {e}")
            output_text = "CORRECT" # Fallback

        flag, sid, corrected = parse_model_output(output_text)

        if flag == 0:
            line = f"{text_id} 0 -1 NA"
        else:
            corrected_escaped = escape_for_double_quotes(corrected)
            line = f'{text_id} 1 {sid} "{corrected_escaped}"'

        f.write(line + "\n")
        print(line)
        print("text_id:", text_id)
        print("error_sentence_id:", item["error_sentence_id"], "True Positive:", sid == item["error_sentence_id"])

print("Saved to:", out_path)

ms-test-17 1 4 "| Mycobacterium tuberculosis is suspected."
text_id: ms-test-17
error_sentence_id: 4 True Positive: True
ms-test-22 0 -1 NA
text_id: ms-test-22
error_sentence_id: -1 True Positive: True
ms-test-29 1 21 "| Diffuse cutaneous systemic scleroderma is suspected."
text_id: ms-test-29
error_sentence_id: 21 True Positive: True
ms-test-43 1 6 "Hemoglobin 14 g/dL"
text_id: ms-test-43
error_sentence_id: 4 True Positive: False
ms-test-45 1 4 "| The patient is diagnosed with polyarteritis nodosa."
text_id: ms-test-45
error_sentence_id: 4 True Positive: True
ms-test-53 1 13 "Patient is diagnosed with Zollinger-Ellison Syndrome."
text_id: ms-test-53
error_sentence_id: 13 True Positive: True
ms-test-67 1 15 "| The patient's severe memory impairment, with recall of 0/3 words after 5 minutes, along with fluctuating cognition and motor symptoms, suggests a diagnosis of Lewy Body Dementia rather than Alzheimer dementia."
text_id: ms-test-67
error_sentence_id: 15 True Positive: True
ms-test